# Reinforcement learning from scratch (numpy only)

Four experiments, each matching a section of `notes.md`:

1. **Value iteration and policy iteration** on a 4x4 gridworld (notes §3):
   the two agree, and value iteration's error shrinks by a factor gamma per sweep.
2. **Monte Carlo vs TD(0)** evaluation of the two-state loop chain (notes §4):
   both converge to V(s1) = 0.818, TD with lower variance per update.
3. **Q-learning vs SARSA** on a cliff (notes §4): SARSA learns the safe path,
   Q-learning the edge path.
4. **REINFORCE on a bandit** with and without a baseline (notes §5): same
   expected gradient, lower variance with the baseline.

Everything runs in a few seconds on CPU.

In [ ]:

import numpy as np
rng = np.random.default_rng(0)



## 1. Gridworld: value iteration vs policy iteration

4x4 grid, start anywhere, goal in the bottom-right corner (terminal, reward +1
on entering). Every other step costs -0.04. Actions: up/down/left/right, with
80% intended move and 10% each of the two perpendicular slips. Bumping a wall
keeps you in place.

In [ ]:

N = 4
GOAL = (N - 1, N - 1)
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # up, down, left, right
GAMMA = 0.9
STEP_COST = -0.04

def move(s, a):
    r, c = s[0] + a[0], s[1] + a[1]
    return s if not (0 <= r < N and 0 <= c < N) else (r, c)

def transitions(s, ai):
    """List of (prob, next_state, reward) for taking action index ai in s."""
    if s == GOAL:
        return [(1.0, s, 0.0)]
    a = ACTIONS[ai]
    perp = [(a[1], a[0]), (-a[1], -a[0])]
    out = []
    for p, d in [(0.8, a), (0.1, perp[0]), (0.1, perp[1])]:
        s2 = move(s, d)
        out.append((p, s2, 1.0 if s2 == GOAL else STEP_COST))
    return out

STATES = [(r, c) for r in range(N) for c in range(N)]

def q_from_v(V):
    Q = {}
    for s in STATES:
        for ai in range(4):
            Q[s, ai] = sum(p * (rew + GAMMA * V[s2]) for p, s2, rew in transitions(s, ai))
    return Q

def value_iteration(tol=1e-10, keep=None):
    V = {s: 0.0 for s in STATES}
    history = []
    while True:
        Q = q_from_v(V)
        V_new = {s: max(Q[s, ai] for ai in range(4)) for s in STATES}
        delta = max(abs(V_new[s] - V[s]) for s in STATES)
        V = V_new
        history.append(dict(V))
        if delta < tol:
            return V, history

V_vi, hist = value_iteration()
print("value iteration sweeps:", len(hist))
# the contraction theorem bounds the distance to the fixed point V*, sweep by sweep
dist = [max(abs(Vk[s] - V_vi[s]) for s in STATES) for Vk in hist[:-1]]
ratios = [dist[i + 1] / dist[i] for i in range(len(dist) - 1) if dist[i] > 1e-8]
print("largest ratio of successive distances to V*:", round(max(ratios), 3),
      "(the theorem guarantees at most gamma = 0.9)")



The theorem is a bound on the distance to the fixed point: every sweep
shrinks it by a factor of at most gamma. The observed ratios never exceed
0.9, and on this grid they are usually well below it.

In [ ]:

def policy_iteration():
    pi = {s: 0 for s in STATES}
    while True:
        # evaluate: solve the linear system (I - gamma P_pi) V = R_pi
        idx = {s: i for i, s in enumerate(STATES)}
        A = np.eye(len(STATES)); b = np.zeros(len(STATES))
        for s in STATES:
            for p, s2, rew in transitions(s, pi[s]):
                A[idx[s], idx[s2]] -= GAMMA * p
                b[idx[s]] += p * rew
        v = np.linalg.solve(A, b)
        V = {s: v[idx[s]] for s in STATES}
        # improve
        Q = q_from_v(V)
        pi_new = {s: max(range(4), key=lambda ai: Q[s, ai]) for s in STATES}
        if pi_new == pi:
            return V, pi
        pi = pi_new

V_pi, pi_star = policy_iteration()
print("max |V_vi - V_pi| =", max(abs(V_vi[s] - V_pi[s]) for s in STATES))
arrows = {0: "^", 1: "v", 2: "<", 3: ">"}
for r in range(N):
    print(" ".join("G" if (r, c) == GOAL else arrows[pi_star[r, c]] for c in range(N)))



Both methods land on the same value function (difference at floating-point
level) and the same policy: every arrow points toward the goal.

## 2. Monte Carlo vs TD(0) on the loop chain

From s1: stay with probability 0.5, move to s2 with probability 0.5, reward
0 either way. From s2: terminal with reward 1. gamma = 0.9, so the exact
value is V(s1) = 0.45/0.55 = 0.818.

In [ ]:

G = 0.9
def episode():
    """Returns the list of (state, reward) pairs, states 0 = s1, 1 = s2."""
    traj = []
    s = 0
    while True:
        if s == 0:
            traj.append((0, 0.0))
            s = 0 if rng.random() < 0.5 else 1
        else:
            traj.append((1, 1.0))
            return traj

def mc_estimate(n_episodes, alpha=0.05):
    V = np.zeros(2)
    for _ in range(n_episodes):
        traj = episode()
        Gt = 0.0
        for s, r in reversed(traj):
            Gt = r + G * Gt
            V[s] += alpha * (Gt - V[s])
    return V

def td_estimate(n_episodes, alpha=0.05):
    V = np.zeros(3)   # index 2 = terminal, value 0
    for _ in range(n_episodes):
        traj = episode()
        for i, (s, r) in enumerate(traj):
            s2 = traj[i + 1][0] if i + 1 < len(traj) else 2
            V[s] += alpha * (r + G * V[s2] - V[s])
    return V[:2]

runs_mc = np.array([mc_estimate(2000)[0] for _ in range(20)])
runs_td = np.array([td_estimate(2000)[0] for _ in range(20)])
print(f"exact V(s1) = {0.45/0.55:.4f}")
print(f"MC : mean {runs_mc.mean():.4f}, std across runs {runs_mc.std():.4f}")
print(f"TD : mean {runs_td.mean():.4f}, std across runs {runs_td.std():.4f}")



Both are centered on 0.818. TD's spread across runs is smaller: its target
uses one reward plus a value estimate instead of a whole episode's return.

## 3. Q-learning vs SARSA on the cliff

4 rows x 8 columns. Start bottom-left, goal bottom-right, the cells between
them on the bottom row are the cliff: stepping on one costs -100 and sends
the agent back to the start. Every other step costs -1. Epsilon-greedy with
epsilon = 0.1 during training for both algorithms.

In [ ]:

ROWS, COLS = 4, 8
START, GOALC = (ROWS - 1, 0), (ROWS - 1, COLS - 1)
CLIFF = {(ROWS - 1, c) for c in range(1, COLS - 1)}

def cliff_step(s, ai):
    a = ACTIONS[ai]
    r, c = min(max(s[0] + a[0], 0), ROWS - 1), min(max(s[1] + a[1], 0), COLS - 1)
    s2 = (r, c)
    if s2 in CLIFF:
        return START, -100.0, False
    return s2, -1.0, s2 == GOALC

def eps_greedy(Q, s, eps):
    if rng.random() < eps:
        return rng.integers(4)
    return int(np.argmax(Q[s]))

def train(kind, episodes=600, alpha=0.5, eps=0.1, gamma=1.0):
    Q = np.zeros((ROWS, COLS, 4))
    returns = []
    for _ in range(episodes):
        s = START; a = eps_greedy(Q, s, eps); total = 0.0
        while True:
            s2, r, done = cliff_step(s, a)
            total += r
            a2 = eps_greedy(Q, s2, eps)
            target = r if done else r + gamma * (Q[s2].max() if kind == "qlearning" else Q[s2][a2])
            Q[s][a] += alpha * (target - Q[s][a])
            s, a = s2, a2
            if done:
                break
        returns.append(total)
    return Q, np.array(returns)

def greedy_path(Q):
    s = START; path = [s]
    for _ in range(50):
        s, _, done = cliff_step(s, int(np.argmax(Q[s])))
        path.append(s)
        if done:
            break
    return path

def show(Q, title):
    path = set(greedy_path(Q))
    print(title)
    for r in range(ROWS):
        row = ""
        for c in range(COLS):
            cell = (r, c)
            row += "C" if cell in CLIFF else "S" if cell == START else "G" if cell == GOALC else "*" if cell in path else "."
        print("  " + row)

Q_ql, ret_ql = train("qlearning")
Q_sa, ret_sa = train("sarsa")
show(Q_ql, "Q-learning greedy path (hugs the cliff):")
show(Q_sa, "SARSA greedy path (keeps its distance):")
print(f"mean return over the last 100 training episodes, with exploration on:")
print(f"  Q-learning {ret_ql[-100:].mean():7.1f}   SARSA {ret_sa[-100:].mean():7.1f}")



Q-learning finds the shortest path, which is optimal for an agent that never
slips. SARSA, evaluating the policy that actually explores, learns a path
that stays away from the cliff and earns a better return *while exploring*.

## 4. REINFORCE on a bandit: the baseline lowers variance

Five arms with Bernoulli rewards of probability [0.1, 0.3, 0.5, 0.7, 0.9].
Softmax policy over five logits. At a fixed policy we draw many single-sample
gradient estimates, with and without the baseline, and compare their means
(should agree) and their variances (should not).

In [ ]:

P_ARMS = np.array([0.1, 0.3, 0.5, 0.7, 0.9])

def softmax(z):
    z = z - z.max(); e = np.exp(z); return e / e.sum()

def grad_samples(theta, n, baseline):
    pi = softmax(theta)
    grads = np.zeros((n, 5))
    for i in range(n):
        a = rng.choice(5, p=pi)
        r = float(rng.random() < P_ARMS[a])
        onehot = np.zeros(5); onehot[a] = 1
        grads[i] = (r - baseline) * (onehot - pi)
    return grads

theta = np.zeros(5)
pi0 = softmax(theta)
true_b = (pi0 * P_ARMS).sum()          # expected reward under the current policy
g_plain = grad_samples(theta, 20000, 0.0)
g_base = grad_samples(theta, 20000, true_b)
print("mean gradient, no baseline :", np.round(g_plain.mean(0), 3))
print("mean gradient, baseline    :", np.round(g_base.mean(0), 3))
print(f"total variance, no baseline: {g_plain.var(0).sum():.4f}")
print(f"total variance, baseline   : {g_base.var(0).sum():.4f}")



The two mean gradients agree (the baseline lemma), pointing up for the good
arms and down for the bad ones, and the baseline cuts the estimator's
variance substantially.

Finally, run REINFORCE with the baseline and watch the policy concentrate on
the best arm.

In [ ]:

theta = np.zeros(5); avg_r = 0.0; lr = 0.1
for t in range(1, 3001):
    pi = softmax(theta)
    a = rng.choice(5, p=pi)
    r = float(rng.random() < P_ARMS[a])
    avg_r += (r - avg_r) / t                # running mean reward as the baseline
    onehot = np.zeros(5); onehot[a] = 1
    theta += lr * (r - avg_r) * (onehot - pi)
print("final policy:", np.round(softmax(theta), 3))
print("probability on the best arm:", round(float(softmax(theta)[4]), 3))
